# 🛠️ Notebook 2: Movie Ticket Booking — Implementation + Concurrency

## 🛠️ Setup

```bash
cd 07-object-oriented-design/movie-ticket-booking
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
import itertools, threading, time, random

class SeatStatus(Enum):
    FREE = 0; HELD = 1; BOOKED = 2

@dataclass
class Seat:
    row: str
    number: int
    status: SeatStatus = SeatStatus.FREE
    @property
    def id(self): return f'{self.row}{self.number}'

@dataclass
class Show:
    id: int
    movie: str
    seats: dict      # seat_id -> Seat
    price: float
    _lock: threading.Lock = field(default_factory=threading.Lock)

@dataclass
class Booking:
    id: int
    user: str
    show: Show
    seats: list
    total: float
    confirmed: bool = False

_bid = itertools.count(1)

class BookingService:
    def hold_and_book(self, show: Show, user: str, seat_ids: list) -> Booking:
        with show._lock:  # critical section: check + update together
            chosen = [show.seats[sid] for sid in seat_ids]
            if any(s.status != SeatStatus.FREE for s in chosen):
                raise RuntimeError(f'{user}: seat already taken')
            for s in chosen: s.status = SeatStatus.BOOKED
        return Booking(next(_bid), user, show, chosen, show.price*len(chosen), confirmed=True)


## Build a show + try to book the same seat twice

In [ ]:
def make_show():
    seats = {}
    for r in 'AB':
        for n in range(1,6):
            s = Seat(r, n); seats[s.id] = s
    return Show(1, 'Inception', seats, price=12.5)

show = make_show()
svc = BookingService()
b1 = svc.hold_and_book(show, 'ada', ['A1','A2'])
print('ada booked:', [s.id for s in b1.seats], 'total $', b1.total)

try:
    svc.hold_and_book(show, 'grace', ['A2','A3'])
except RuntimeError as e:
    print('expected:', e)


## Concurrency: many users racing for the same seat

In [ ]:
show = make_show()
winners = []
losers = []

def try_book(user):
    time.sleep(random.random()*0.01)
    try:
        b = svc.hold_and_book(show, user, ['A1'])
        winners.append(user)
    except Exception:
        losers.append(user)

threads = [threading.Thread(target=try_book, args=(f'u{i}',)) for i in range(20)]
for t in threads: t.start()
for t in threads: t.join()

print('winner (exactly one):', winners)
print('losers :', len(losers))
assert len(winners) == 1


### Why this matters
- Without the lock, `check-then-set` becomes a race: two users see the seat free and both 'book' it.
- In a distributed system you'd replace the `threading.Lock` with a **database transaction** or a **Redis lock**.